# Introduction to AI Agents
### Practice Notebook

**Assumed pre-installed libraries:** none required (pure Python standard library)

This notebook has no live LLM API calls wired in. Instead, we simulate the
"reasoning" step of an agent with simple rule-based stand-ins, so you can see
the *structure* of the agent loop clearly before layering a real LLM on top
of it. Wherever you see `# TODO: replace with a real LLM call`, that's where
a production system would call Claude (or another model) instead.


In [1]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HF_TOKEN")
client = InferenceClient(api_key=HF_TOKEN)

c:\STUDY MATERIAL\CODING\Python Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. A simple LLM call vs. an agent

A single LLM call is:

```text
User → LLM → Answer
```

An agent is a loop:

```text
Goal → LLM → Tool → Observation → LLM → Tool → Observation → Answer
```

The important difference is **interaction with an external environment**. The agent can obtain information that was not present in the original prompt.

In [2]:
def single_llm_call(user_request: str) -> str:
    """A single LLM call: one input, one output, done. No ability to check
    its own work or gather more information before answering.
    """
    response = client.chat_completion(
        model="Qwen/Qwen2.5-Coder-32B-Instruct",
        messages=[{"role": "user", "content": user_request}],
        max_tokens=200,
    )
    return response.choices[0].message.content

print(single_llm_call("What's the cheapest flight from Bengaluru to Mumbai next Friday?"))


To find the cheapest flight from Bengaluru (BLR) to Mumbai (BOM) for the coming Friday, you can use several online travel booking websites and apps that compare prices across different airlines and travel agencies. Here are a few popular ones you might consider:

1. **MakeMyTrip**: One of the leading Indian travel portals, offering flight search and booking services.
2. **Yatra**: Another prominent platform in India, known for competitive fares and easy-to-use interface.
3. **Expedia**: A global travel site that covers many airlines and offers good deals.
4. **Goibibo**: Known for its budget-friendly options and user-friendly interface.
5. **AirAsia**: Offers direct flights and competitive pricing on most routes.
6. **SpiceJet**: Known for its budget flights, SpiceJet often provides good deals and discounts.

Here’s a general approach to finding the cheapest flight:

- **Log in**: Visit one of these websites and log in.
- **Search


Notice the single call above can only ever *guess* -- it has no way to
actually check flight prices. This is exactly the gap the agent loop below
is designed to close.


## 2. The agent loop: Perceive → Reason → Act

```
        +-------------------------------------+
        |                                     |
        v                                     |
   [PERCEIVE]  ---->  [REASON]  ---->  [ACT]  -+
   (read current       (LLM decides    (call a tool,
    state / last          what to do    take an action)
    tool result)           next)
```

Let's build this loop for a tiny toy task: an agent that needs to find the
cheapest of several flight options. "Perceive" starts as the user's request;
each "act" step calls a (fake) tool; the tool's result becomes the next
thing to "perceive".


In [3]:
# Deterministic fake flight data.
# Deterministic data makes the notebook easier to reproduce.

FLIGHTS = [
    {"airline": "IndiGo", "price": 4200},
    {"airline": "Air India", "price": 5100},
    {"airline": "Vistara", "price": 3900},
    {"airline": "SpiceJet", "price": 4500},
]

# Seat availability for Exercise 1.1
SEAT_AVAILABLE = {
    "Vistara": False,
    "IndiGo": True,
    "SpiceJet": True,
    "Air India": True,
}


def search_flights_tool(origin: str, destination: str) -> list:
    """Fake flight-search API."""
    return FLIGHTS.copy()


def check_seat_availability(airline: str, price: int) -> bool:
    """Fake seat-availability API."""
    return SEAT_AVAILABLE.get(airline, False)


def sort_flights_by_price(flight_options: list) -> list:
    return sorted(flight_options, key=lambda x: x["price"])

In [4]:
import json

def reason_pick_cheapest(flight_options: list) -> dict:
    # 1. Define the prompt variable using flight_options
    prompt = f"""You are a flight selection agent.
Given these flight options: {flight_options}

Select the flight with the minimum price.
Return ONLY a raw JSON object representing that flight with no extra words or markdown.
Example format: {{"airline": "Name", "price": 1000}}"""

    # 2. Call the LLM API using client
    response = client.chat_completion(
        model="Qwen/Qwen2.5-Coder-32B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
    )
    
    # 3. Clean and parse the response into a Python dictionary
    raw_content = response.choices[0].message.content.strip()
    clean_json = raw_content.replace("```json", "").replace("```", "").strip()
    return json.loads(clean_json)

In [5]:
def run_agent(origin: str, destination: str):
    print(f"[PERCEIVE] Goal: find cheapest flight {origin} -> {destination}")

    print("[ACT] Calling search_flights_tool(...)")
    flight_options = search_flights_tool(origin, destination)
    print(f"[PERCEIVE] Observed tool result: {flight_options}")

    print("[REASON] Deciding which option is best...")
    best = reason_pick_cheapest(flight_options)

    # Access dictionary keys directly instead of best[0] / best[1]
    print(f"[ACT] Final answer: book {best['airline']} at Rs.{best['price']}")
    return best

run_agent("Bengaluru", "Mumbai")

[PERCEIVE] Goal: find cheapest flight Bengaluru -> Mumbai
[ACT] Calling search_flights_tool(...)
[PERCEIVE] Observed tool result: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
[REASON] Deciding which option is best...
[ACT] Final answer: book Vistara at Rs.3900


{'airline': 'Vistara', 'price': 3900}

**Exercise 1.1:** The loop above only runs one perceive-reason-act cycle. Add
a second cycle: after picking the cheapest flight, call a second fake tool
`check_seat_availability(airline, price)` (write it yourself -- have it
randomly return True/False), and if it returns `False`, go back and pick the
*next* cheapest option instead. This is the essence of a *multi-step* agent
loop instead of a single pass.


In [6]:
import random
FLIGHTS = [
    {"airline": "IndiGo", "price": 4200},
    {"airline": "Air India", "price": 5100},
    {"airline": "Vistara", "price": 3900},
    {"airline": "SpiceJet", "price": 4500},
]

def search_flights_tool(origin: str, destination: str) -> list:
    """Fake flight-search API."""
    return FLIGHTS.copy()

def check_seat_availability(airline: str, price: int) -> bool:
    """Fake seat-availability API as requested by Exercise 1.1 (returns random True/False)."""
    return random.choice([True, False])

def sort_flights_by_price(flight_options: list) -> list:
    """Sort flights from lowest to highest price."""
    return sorted(flight_options, key=lambda x: x["price"])

def run_agent_multistep(origin: str, destination: str):
    print(f"[PERCEIVE] Goal: find cheapest available flight {origin} -> {destination}")
    print("[ACT] Calling search_flights_tool(...)")
    flight_options = search_flights_tool(origin, destination)
    print(f"[PERCEIVE] Observed tool result: {flight_options}")

    # Step 3: Reason - Sort options by price
    print("[REASON] Sorting flight options by price...")
    sorted_flights = sort_flights_by_price(flight_options)

    # Step 4: Multi-step Loop (Perceive -> Reason -> Act until a seat is available)
    for option in sorted_flights:
        airline = option["airline"]
        price = option["price"]

        print(f"[REASON] Considering cheapest remaining option: {airline} at Rs.{price}")
        print(f"[ACT] Calling check_seat_availability('{airline}', {price})...")
        
        # Check seat availability tool call
        is_available = check_seat_availability(airline, price)
        print(f"[PERCEIVE] Observed seat availability: {is_available}")

        if is_available:
            print(f"[REASON] Seat available! Decision finalized.")
            print(f"[ACT] Final answer: Book {airline} for Rs.{price}")
            return option
        else:
            print(f"[REASON] No seats on {airline}. Backtracking to pick the next option...\n")

    print("[ACT] Final answer: No available flights found.")
    return None

# Test the multi-step agent loop
run_agent_multistep("Bengaluru", "Mumbai")

[PERCEIVE] Goal: find cheapest available flight Bengaluru -> Mumbai
[ACT] Calling search_flights_tool(...)
[PERCEIVE] Observed tool result: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
[REASON] Sorting flight options by price...
[REASON] Considering cheapest remaining option: Vistara at Rs.3900
[ACT] Calling check_seat_availability('Vistara', 3900)...
[PERCEIVE] Observed seat availability: False
[REASON] No seats on Vistara. Backtracking to pick the next option...

[REASON] Considering cheapest remaining option: IndiGo at Rs.4200
[ACT] Calling check_seat_availability('IndiGo', 4200)...
[PERCEIVE] Observed seat availability: False
[REASON] No seats on IndiGo. Backtracking to pick the next option...

[REASON] Considering cheapest remaining option: SpiceJet at Rs.4500
[ACT] Calling check_seat_availability('SpiceJet', 4500)...
[PERCEIVE] Observed seat availability: True
[REASON

{'airline': 'SpiceJet', 'price': 4500}

## 3. The ReAct pattern: interleaving reasoning and acting

ReAct (Yao et al., 2022) interleaves explicit reasoning traces with actions:
the model writes out *why* it's about to do something, then does it, then
reasons about the result, then acts again. Let's make the reasoning trace
visible as text, the way a real ReAct-style LLM transcript would look.


In [8]:
def react_agent_transcript(origin: str, destination: str):
    transcript = []

    transcript.append(f"Thought: I need to find flights from {origin} to {destination}. "
                       f"I should call the flight search tool.")
    transcript.append("Action: search_flights_tool(origin, destination)")
    flights = search_flights_tool(origin, destination)
    transcript.append(f"Observation: {flights}")

    transcript.append("Thought: Now I should compare prices and pick the cheapest one.")
    best = reason_pick_cheapest(flights)
    transcript.append(f"Action: select_flight({best['airline']})")
    transcript.append(f"Observation: {best['airline']} confirmed available at Rs.{best['price']}")

    transcript.append(f"Thought: I have enough information to answer the user now.")
    transcript.append(f"Final Answer: Book {best['airline']} for Rs.{best['price']}, the cheapest option found.")

    return transcript

for line in react_agent_transcript("Bengaluru", "Mumbai"):
    print(line)

Thought: I need to find flights from Bengaluru to Mumbai. I should call the flight search tool.
Action: search_flights_tool(origin, destination)
Observation: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
Thought: Now I should compare prices and pick the cheapest one.
Action: select_flight(Vistara)
Observation: Vistara confirmed available at Rs.3900
Thought: I have enough information to answer the user now.
Final Answer: Book Vistara for Rs.3900, the cheapest option found.


**Exercise 1.2:** Notice every `Thought:` line above is hard-coded text, not
actually generated reasoning -- the *agent* isn't really "deciding" anything,
we are. Identify: which lines would change if the tool returned a different
result (e.g., an empty flight list, meaning no flights are available)? Add a
branch to `react_agent_transcript` that produces a sensible transcript for
that case.



In [9]:
def react_agent_transcript(origin: str, destination: str, flights_override: list = None):
    transcript = []

    transcript.append(f"Thought: I need to find flights from {origin} to {destination}. "
                      f"I should call the flight search tool.")
    transcript.append("Action: search_flights_tool(origin, destination)")
    
    
    flights = flights_override if flights_override is not None else search_flights_tool(origin, destination)
    transcript.append(f"Observation: {flights}")

    
    if not flights:
        transcript.append("Thought: The search returned no flight options. I cannot compare prices or select a flight.")
        transcript.append("Action: notify_user_no_flights()")
        transcript.append("Observation: User informed that no options are available.")
        transcript.append("Thought: I have enough information to inform the user about the lack of options.")
        transcript.append(f"Final Answer: Sorry, no flights were found from {origin} to {destination}.")
        return transcript

    transcript.append("Thought: Now I should compare prices and pick the cheapest one.")
    best = reason_pick_cheapest(flights)
    
    transcript.append(f"Action: select_flight({best['airline']})")
    transcript.append(f"Observation: {best['airline']} confirmed available at Rs.{best['price']})")

    transcript.append("Thought: I have enough information to answer the user now.")
    transcript.append(f"Final Answer: Book {best['airline']} for Rs.{best['price']}, the cheapest option found.")

    return transcript

# Test 1: Normal case
print("--- NORMAL CASE TRANSCRIPT ---")
for line in react_agent_transcript("Bengaluru", "Mumbai"):
    print(line)

# Test 2: Empty flights case
print("\n--- EMPTY FLIGHTS CASE TRANSCRIPT ---")
for line in react_agent_transcript("Bengaluru", "Mumbai", flights_override=[]):
    print(line)

--- NORMAL CASE TRANSCRIPT ---
Thought: I need to find flights from Bengaluru to Mumbai. I should call the flight search tool.
Action: search_flights_tool(origin, destination)
Observation: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
Thought: Now I should compare prices and pick the cheapest one.
Action: select_flight(Vistara)
Observation: Vistara confirmed available at Rs.3900)
Thought: I have enough information to answer the user now.
Final Answer: Book Vistara for Rs.3900, the cheapest option found.

--- EMPTY FLIGHTS CASE TRANSCRIPT ---
Thought: I need to find flights from Bengaluru to Mumbai. I should call the flight search tool.
Action: search_flights_tool(origin, destination)
Observation: []
Thought: The search returned no flight options. I cannot compare prices or select a flight.
Action: notify_user_no_flights()
Observation: User informed that no options are availa

**Exercise 1.3:** Compare `run_agent` (Part 2) and `react_agent_transcript`(Part 3) -- what's actually different about the two implementations,structurally? (Hint: it's not the underlying logic, which is the same --it's what's made *visible* and *inspectable*.) Why might that visibility
matter when debugging a real agent that misbehaves?


#### Comparison: `run_agent` vs. `react_agent_transcript`

#### What is structurally different?
* **`run_agent` (Part 2):** Just runs the Python functions behind the scenes and prints basic status updates. Once it finishes, the intermediate steps are gone.
* **`react_agent_transcript` (Part 3):** Builds an explicit step-by-step history log using the **ReAct** framework:
  * **`Thought:`** Explains *why* it's taking an action.
  * **`Action:`** Shows the exact tool being called.
  * **`Observation:`** Shows the output returned by that tool.

The underlying logic is identical, but `react_agent_transcript` creates a visible text trace of the entire decision-making process.


#### Why does this visibility matter when debugging?

* **Easy Bug Identification:** If the agent makes a mistake, the transcript shows you exact steps so you can see if it had a bad idea (Thought), picked the wrong button or tool to use (Action), or got wrong information back from the system (Observation).


* **Prevents Loops:** Showing the transcript to the AI reminds it of what it already tried, stopping it from doing the exact same useless step over and over again.


* **Receipt:** It gives you a clear receipt of every choice the agent made, so you can check its work and make sure it made sense before trusting its decision.

## 4. When *not* to use an agent

Agents add latency (multiple LLM calls instead of one), cost, and
unpredictability. Not every task needs a loop.

**Exercise 1.4 (discussion, no code):** For each task below, decide whether
it's better solved with (a) a single LLM call, (b) a fixed multi-step
workflow (Day 3 topic, predefined steps, no loop), or (c) a full
perceive-reason-act agent loop -- and justify your answer in one sentence:

1. Translate a sentence from English to Hindi.
2. Answer "What's on my calendar tomorrow?" by checking a calendar API.
3. "Plan and book my entire 5-day trip to Goa, adjusting for weather and
   budget as you go, and re-book anything that falls through."
4. Classify a support ticket as "billing," "technical," or "other."



### Choosing the Right Tool

1. **Translate English to Hindi**
   * **Answer:** (a) Single LLM call
   * **Why:** Translation is a direct text transformation task that needs no outside tools, extra steps, or loops.

2. **"What's on my calendar tomorrow?"**
   * **Answer:** (b) Fixed multi-step workflow
   * **Why:** This follows a strict 2-step pipeline (get tomorrow's date $\rightarrow$ call the calendar API) that never needs to change or loop.

3. **Plan and book a 5-day Goa trip with weather/budget adjustments and re-booking**
   * **Answer:** (c) Full agent loop
   * **Why:** This task is dynamic and unpredictable, requiring continuous checking, decision-making, and backtracking when plans fail.

4. **Classify a support ticket as "billing," "technical," or "other"**
   * **Answer:** (a) Single LLM call
   * **Why:** The LLM can read the ticket text and categorize it immediately in one step without calling external APIs.